# Lab 1 — Crawling & Extraction de Connaissances

**Cours :** Web Mining & Sémantique  
**Auteurs :** Bilal Gougis & Chadi Al Kerdi  
**Domaine :** Biologie des Céphalopodes (Poulpes)

On va crawler des pages web sur les poulpes, nettoyer le contenu et extraire des entités/relations avec spaCy.

In [ ]:
%pip install -q spacy pandas tqdm beautifulsoup4 requests rdflib SPARQLWrapper pykeen scikit-learn matplotlib gradio owlready2
%pip uninstall torch -y
%pip install torch==2.3.1 --index-url https://download.pytorch.org/whl/cpu
!python -m spacy download fr_core_news_md


In [ ]:
import json, time, hashlib, re, os, random, warnings
from collections import deque, defaultdict, Counter
from typing import List, Tuple, Dict, Optional
from urllib.robotparser import RobotFileParser
from urllib.parse import quote, urlparse

import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import spacy
from bs4 import BeautifulSoup
from tqdm import tqdm
from rdflib import Graph, Namespace, URIRef, Literal, BNode
from rdflib.namespace import RDF, RDFS, OWL, XSD
from SPARQLWrapper import SPARQLWrapper, JSON

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("imports ok")


In [ ]:
CRAWLER_OUTPUT   = "crawler_output.jsonl"
ENTITIES_OUTPUT  = "extracted_knowledge.csv"
RELATIONS_OUTPUT = "extracted_knowledge_relations.csv"
MIN_WORD_COUNT   = 150
CRAWL_DELAY      = 1.0

HEADERS = {
    "User-Agent":      "Mozilla/5.0 (academic research bot; student@lab.fr)",
    "Accept-Language": "fr-FR,fr;q=0.9",
}

NOUVEAUX_URLS  = [
    "https://animaldiversity.org/accounts/Octopus_vulgaris/",
    "https://www.britannica.com/animal/octopus-mollusk",
    "https://www.nationalgeographic.com/animals/invertebrates/facts/octopus",
    "https://www.marinespecies.org/aphia.php?p=taxdetails&id=11707",
    "https://www.marinespecies.org/aphia.php?p=taxdetails&id=405243",
    "https://www.marinespecies.org/aphia.php?p=taxdetails&id=138127",
    "https://www.frontiersin.org/articles?query=octopus",
    "https://www.softschools.com/facts/animals/octopus_facts/92/",
    "https://reporterre.net/Tout-savoir-sur-le-poulpe-en-5-faits-etonnants",
    "https://naturdive.com/actions/les-poulpes/",
    "https://doris.ffessm.fr/Especes/Octopus-vulgaris-Poulpe-commun-847",
    "https://www.larousse.fr/encyclopedie/vie-sauvage/pieuvre_ou_poulpe/184019",
    "https://www.peche.com/article/32277/le-poulpe-ce-cephalopode-intelligent",
    "https://www.salamandre.org/article/poulpe-intelligence-humains-neuf-cerveaux/",
    "https://www.ouest-france.fr/leditiondusoir/2021-06-02/pourquoi-lintelligence-des-poulpes-fascine-les-scientifiques-fd94b7ce-6a98-42ae-aa51-3d1465a1b68a",
    "https://www.sorbonne-universite.fr/dossiers/sciences-de-la-mer/laure-bonnaud-ponticelli-la-capacite-danalyse-du-poulpe-est-phenomenale",
    "https://www.cortex-mag.net/lintelligence-distribuee-de-la-pieuvre/",
    "https://www.salamandre.org/article/a-travers-les-oceans-des-poulpes-aux-couleurs-extravagantes/",
    "https://www.fishipedia.fr/fr/mollusques/famille/octopodidae",
    "https://www.fishipedia.fr/fr/mollusques/callistoctopus-macropus",
    "https://www.fishipedia.fr/fr/mollusques/hapalochlaena-fasciata",
    "https://www.fishipedia.fr/fr/mollusques/hapalochlaena-lunulata",
    "https://www.fishipedia.fr/fr/mollusques/hapalochlaena-maculosa",
    "https://www.fishipedia.fr/fr/mollusques/octopus-cyanea",
    "https://www.fishipedia.fr/fr/mollusques/octopus-vulgaris",
    "https://www.fishipedia.fr/fr/mollusques/thaumoctopus-mimicus",
    "https://junior.universalis.fr/encyclopedie/pieuvre",
]

# URLs de crawling
WIKIPEDIA_TITLES = [
    "Octopus vulgaris",
    "Enteroctopus dofleini",
    "Octopus cyanea",
    "Eledone cirrhosa",
    "Amphioctopus marginatus",
    "Callistoctopus macropus",
    "Tremoctopus",
    "Hapalochlaena",
    "Wunderpus photogenicus",
    "Pieuvre à anneaux bleus",
    "Pieuvre",
    "Céphalopode",
    "Chromatophore",
    "Intelligence des céphalopodes",
    "Bec des céphalopodes",
    "Céphalopodes utilisés en cuisine",
]

AQUAPORTAIL_URLS = [
    "https://www.aquaportail.com/especes/taxonomie/genre/533/octopus",
    "https://www.aquaportail.com/fiche-invertebre-3886-octopus-vulgaris.html",
    "https://www.aquaportail.com/dictionnaire/definition/4564/poulpe",
]

MARINESPECIES_URLS = [
    "https://www.marinespecies.org/aphia.php?p=taxdetails&id=140605",
    "https://www.marinespecies.org/aphia.php?p=taxdetails&id=140612",
    "https://www.marinespecies.org/aphia.php?p=taxdetails&id=140613",
    "https://www.marinespecies.org/aphia.php?p=taxdetails&id=342218",
    "https://www.marinespecies.org/aphia.php?p=taxdetails&id=215450",
]

FISHIPEDIA_URLS = [
    "https://www.fishipedia.fr/fr/mollusques/octopus-vulgaris",
    "https://www.fishipedia.fr/fr/mollusques/octopus-cyanea",
    "https://www.fishipedia.fr/fr/mollusques/type/poulpe",
]

GUIDEDESESPECES_URLS = [
    "https://www.guidedesespeces.org/fr/poulpe",
    "https://www.guidedesespeces.org/fr/mollusques",
]

# NLP
KEEP_LABELS   = {"PER", "ORG", "LOC", "MISC"}
SUBJECT_DEPS  = {"nsubj", "nsubjpass"}
OBJECT_DEPS   = {"obj", "dobj", "iobj", "obl", "pobj", "attr"}
VERBES_EXCLUS = {"=", "être", "avoir", "pouvoir", "falloir", "aller"}

ESPECES = {
    "octopus vulgaris", "enteroctopus dofleini", "octopus cyanea",
    "eledone cirrhosa", "hapalochlaena", "amphioctopus marginatus",
    "callistoctopus macropus", "tremoctopus", "wunderpus photogenicus",
    "pieuvre à anneaux bleus", "pieuvre commune", "poulpe commun",
    "pieuvre", "poulpe", "céphalopode", "octopode", "chromatophore",
}

print("✓ Configuration chargée")


## Phase 1 : Crawling

On va chercher des pages sur les poulpes depuis Wikipedia FR, marinespecies, guidedesespeces et d'autres sites spécialisés.

In [ ]:
# PHASE 1 — CRAWLING


def check_robots_txt(url):
    # Vérifie si le crawling est autorisé par robots.txt.
    try:
        parsed = urlparse(url)
        robots_url = f"{parsed.scheme}://{parsed.netloc}/robots.txt"
        rp = RobotFileParser()
        rp.set_url(robots_url)
        rp.read()
        allowed = rp.can_fetch(HEADERS["User-Agent"], url)
        return allowed
    except Exception:
        return True  # En cas d'erreur, on autorise (robots.txt absent)

def make_page(url, title, text, source):
    # Créer un objet page standardisé.
    url = quote(url, safe=":/?#[]@!$&'()*+,;=%")
    return {
        "url":          url,
        "title":        title,
        "text":         text,
        "word_count":   len(text.split()),
        "source":       source,
        "content_hash": hashlib.md5(text.encode()).hexdigest(),
    }

def clean_html(soup):
    # Supprimer les balises inutiles et retourner le texte propre.
    for tag in soup(["nav","footer","script","style","header","aside","form"]):
        tag.decompose()
    main = soup.find("main") or soup.find("article") or soup.find("body")
    text = (main or soup).get_text(separator=" ", strip=True)
    return re.sub(r"\s+", " ", text).strip()

def fetch_wikipedia(titles):
    print("\n=== Wikipedia FR ===")
    pages = []
    for title in titles:
        try:
            resp = requests.get(
                "https://fr.wikipedia.org/w/api.php",
                params={"action":"query","titles":title,"prop":"extracts",
                        "explaintext":True,"format":"json","redirects":True},
                headers=HEADERS, timeout=20
            )
            resp.raise_for_status()
            page = next(iter(resp.json()["query"]["pages"].values()))
            text = page.get("extract","")
            if len(text.split()) >= MIN_WORD_COUNT:
                p = make_page(
                    url=f"https://fr.wikipedia.org/wiki/{quote(page.get('title', title), safe='')}",
                    title=page.get("title", title.replace("_"," ")),
                    text=text, source="wikipedia_fr"
                )
                pages.append(p)
                print(f"  ✓ {p['title'][:55]:55} {p['word_count']:>6} mots")
            else:
                print(f"  ✗ {title[:55]} — trop court")
        except Exception as e:
            print(f"  ✗ {title} — {e}")
        time.sleep(CRAWL_DELAY)
    return pages

def fetch_html_pages(urls, source_name):
    print(f"\n=== {source_name} ===")
    pages = []
    for url in urls:
        if not check_robots_txt(url):
            print(f"  ✗ {url} — bloqué par robots.txt")
            continue
        try:
            resp = requests.get(url, headers=HEADERS, timeout=20)
            resp.raise_for_status()
            soup = BeautifulSoup(resp.text, "html.parser")
            text = clean_html(soup)
            if len(text.split()) >= MIN_WORD_COUNT:
                h1    = soup.find("h1")
                title = h1.get_text(strip=True) if h1 else url.split("/")[-1].replace("-"," ").title()
                p = make_page(url=url, title=title, text=text, source=source_name)
                pages.append(p)
                print(f"  ✓ {title[:55]:55} {p['word_count']:>6} mots")
            else:
                print(f"  ✗ {url.split('/')[-1]} — trop court ({len(text.split())} mots)")
        except Exception as e:
            print(f"  ✗ {url} — {e}")
        time.sleep(CRAWL_DELAY)
    return pages

print("\n" + "=" * 60)
print("  CRAWLING POULPES — Multi-sources")
print("=" * 60)

all_pages = []
all_pages += fetch_wikipedia(WIKIPEDIA_TITLES)
all_pages += fetch_html_pages(AQUAPORTAIL_URLS,     "aquaportail")
all_pages += fetch_html_pages(MARINESPECIES_URLS,   "marinespecies")
all_pages += fetch_html_pages(FISHIPEDIA_URLS,      "fishipedia")
all_pages += fetch_html_pages(GUIDEDESESPECES_URLS, "guidedesespeces")
all_pages += fetch_html_pages(NOUVEAUX_URLS, "nv_url")

# Dédupliquer par hash
seen, unique_pages = set(), []
for p in all_pages:
    if p["content_hash"] not in seen:
        seen.add(p["content_hash"])
        unique_pages.append(p)

# Sauvegarder
with open(CRAWLER_OUTPUT, "w", encoding="utf-8") as f:
    for p in unique_pages:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")

# Résumé
print("\n" + "=" * 60)
print(f"  RÉSULTAT : {len(unique_pages)} pages uniques")
print("=" * 60)
for src, count in Counter(p["source"] for p in unique_pages).items():
    mots = sum(p["word_count"] for p in unique_pages if p["source"] == src)
    print(f"  {src:20} {count:2} pages   {mots:,} mots")
print(f"\n  Total mots : {sum(p['word_count'] for p in unique_pages):,}")
print(f"  → {CRAWLER_OUTPUT}")



## Phase 2 : NLP — NER + Extraction de Relations

Ici on va utiliser spaCy pour extraire les entités nommées et les relations sujet-verbe-objet depuis le texte crawlé.

In [ ]:
# PHASE 2 — NLP : NER + EXTRACTION DE RELATIONS

print("\nChargement du modèle spaCy FR...")
try:
    nlp = spacy.load("fr_core_news_md")
    print("  Modèle : fr_core_news_md")
except Exception:
    nlp = spacy.load("fr_core_news_sm")
    print("  Modèle : fr_core_news_sm (fallback)")

def chunk_text(text, max_chars=8000):
    # Découper le texte en chunks pour spaCy.
    chunks, current = [], ""
    for p in [p.strip() for p in text.split("\n") if p.strip()]:
        if len(current) + len(p) > max_chars:
            if current: chunks.append(current.strip())
            current = p
        else:
            current += " " + p
    if current: chunks.append(current.strip())
    return chunks

def extract_entities(doc, source_url):
    # Extraire les entités nommées (NER).
    seen, rows = set(), []
    for ent in doc.ents:
        if ent.label_ not in KEEP_LABELS: continue
        text = re.sub(r"\s+", " ", ent.text.strip())
        if not text or len(text) < 2: continue
        key = (text.lower(), ent.label_)
        if key not in seen:
            seen.add(key)
            rows.append({"entity":text, "label":ent.label_, "source_url":source_url})
    return rows

def find_espece(text):
    # Trouver un nom d'espèce dans un texte.
    text_lower = text.lower()
    for esp in sorted(ESPECES, key=len, reverse=True):
        if esp in text_lower:
            return esp
    return None

def extract_relations(doc, source_url):
    # Extraire les relations sujet(espèce)-verbe-objet.
    relations = []
    for sent in doc.sents:
        sent_text = sent.text.strip()
        if len(sent_text.split()) < 5: continue
        espece = find_espece(sent_text)
        if not espece: continue
        all_entities.append({
            "entity":     espece,
            "label":      "SPECIES",
            "source_url": source_url
        })
        for token in sent:
            if token.pos_ != "VERB": continue
            if token.lemma_ in VERBES_EXCLUS: continue
            for child in token.children:
                if child.dep_ not in OBJECT_DEPS: continue
                subtree = " ".join(t.text for t in child.subtree).strip()
                if len(subtree.split()) < 2: continue
                relations.append({
                    "subject":       espece,
                    "subject_label": "SPECIES",
                    "relation":      token.lemma_,
                    "object":        subtree[:100],
                    "object_label":  child.pos_,
                    "sentence":      sent_text[:200],
                    "source_url":    source_url,
                })
    return relations

print("\n" + "=" * 60)
print("  EXTRACTION NLP (NER + Relations)")
print("=" * 60)

all_entities, all_relations = [], []
for i, page in enumerate(unique_pages):
    print(f"\n[{i+1}/{len(unique_pages)}] {page['title']} ({page['source']})")
    for chunk in chunk_text(page["text"]):
        doc = nlp(chunk)
        all_entities.extend(extract_entities(doc, page["url"]))
        all_relations.extend(extract_relations(doc, page["url"]))
    print(f"  entités: {len(all_entities):,}  relations: {len(all_relations):,}")

# Sauvegarder
ent_df = pd.DataFrame(all_entities).drop_duplicates(subset=["entity","label"])
rel_df = pd.DataFrame(all_relations).drop_duplicates(subset=["subject","relation","object"])

ent_df.to_csv(ENTITIES_OUTPUT,  index=False)
rel_df.to_csv(RELATIONS_OUTPUT, index=False)

print(f"\n→ {ENTITIES_OUTPUT}   ({len(ent_df):,} entités)")
print(f"→ {RELATIONS_OUTPUT}  ({len(rel_df):,} relations)")
print("\n✓ Lab 1 terminé.")